# MicroDuck JumpSpin 360 — Colab A100 Trainer

이 노트북은 실험적 **JumpSpin 360** task를 학습합니다.

- Task: `Mjlab-JumpSpin-Flat-MicroDuck`
- 4096 parallel environments
- 180° → 270° → 360° curriculum
- standing / mid-air / landing reverse curriculum
- 목표 10,000 PPO iterations
- task 자체 checkpoint 간격 100 iterations
- Google Drive 자동 backup + resume

실제 로봇 배포 전에는 simulation rollout과 착지 충격/관절 한계를 반드시 검증하세요.

먼저 **Runtime → Change runtime type → A100 GPU**를 선택하세요.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess
subprocess.run([
    "nvidia-smi",
    "--query-gpu=name,memory.total,driver_version",
    "--format=csv,noheader",
], check=False)

## 1. 설정

In [ ]:
TASK_ID = "Mjlab-JumpSpin-Flat-MicroDuck"
EXPERIMENT_NAME = "jump_spin"
NUM_ENVS = 4096
TARGET_ITERS = 10000
SYNC_INTERVAL = 60
DRIVE_ROOT = "/content/drive/MyDrive/microduck-jump-spin-training"
SMOKE_TEST = True

## 2. 저장소 clone + 설치

In [ ]:
import os, shutil, subprocess

REPO_DIR = "/content/microduck"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.run([
        "git", "clone",
        "https://github.com/namyikim/microduck.git",
        REPO_DIR,
    ], check=True)

subprocess.run(
    ["bash", f"{REPO_DIR}/scripts/setup_colab.sh"],
    check=True,
)

## 3. 학습 시작 / 자동 resume

In [ ]:
import os, subprocess

env = os.environ.copy()
env.update({
    "TASK_ID": TASK_ID,
    "EXPERIMENT_NAME": EXPERIMENT_NAME,
    "NUM_ENVS": str(NUM_ENVS),
    "TARGET_ITERS": str(TARGET_ITERS),
    "SYNC_INTERVAL": str(SYNC_INTERVAL),
    "DRIVE_ROOT": DRIVE_ROOT,
    "SMOKE_TEST": "1" if SMOKE_TEST else "0",
    "WANDB_MODE": env.get("WANDB_MODE", "offline"),
    "REPO_DIR": REPO_DIR,
    "PYTHONUNBUFFERED": "1",
})

print("=== MicroDuck training starting ===", flush=True)
print("Output should appear continuously below this cell.", flush=True)
print(f"Backup log: {DRIVE_ROOT}/training.log", flush=True)

process = subprocess.Popen(
    ["bash", f"{REPO_DIR}/scripts/train_colab.sh"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, process.args)


## 4. Drive checkpoint 확인

In [ ]:
from pathlib import Path
import re

root = Path(DRIVE_ROOT) / "logs" / "rsl_rl" / EXPERIMENT_NAME
items = []
if root.exists():
    for p in root.rglob("model_*.pt"):
        m = re.fullmatch(r"model_(\d+)\.pt", p.name)
        if m:
            items.append((int(m.group(1)), p))
items.sort(key=lambda x: x[0])

if not items:
    print("아직 checkpoint가 없습니다.")
else:
    for n, p in items[-10:]:
        print(f"{n + 1:6d} iterations (approx.)  {p}")
    print("Latest:", items[-1][1])